# LNUQNet: The Model

This notebook implements the LNUQNet model, a combined Convolutional Neural Network (CNN) and Transformer architecture, for lithology classification from well log data. The pipeline is designed to be clean and end-to-end, taking a list of CSV files as input.

In [ ]:
# =========================================================
# LNUQNet: Libraries Used
# =========================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import random
import os

### Main Execution Block

This block orchestrates the entire pipeline:
1.  **Load Data**: Reads all specified CSV files into a single DataFrame.
2.  **Encode Labels**: Converts categorical lithology labels into numerical format.
3.  **Prepare Sliding Windows**: Applies the sliding window technique to create sequences of features and corresponding labels.
4.  **Normalize Features**: Scales feature data using StandardScaler.
5.  **Define Train/Test Wells**: Specifies which wells are used for training and which for testing to ensure a robust well-based split.
6.  **Split Data**: Divides the data into training, validation, and test sets based on well assignments.
7.  **Compute Class Weights**: Calculates class weights to handle potential class imbalance during training.
8.  **Build and Compile Model**: Initializes the LNUQNet model and configures its optimizer, loss function, and metrics.
9.  **Train Model**: Initiates the training process using the prepared data and configuration.

In [ ]:
# =========================================================
# Reproducibility
# =========================================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# =========================================================
# Configuration
# =========================================================

WINDOW_SIZE = 30
BATCH_SIZE = 32
EPOCHS = 100

DEPTH_COL = "Depth.m"
LABEL_COL = "Lithology"

#  LIST OF CSV FILES (USER PROVIDED)
CSV_FILES = [
    "classified_A10.csv", "classified_A15.csv", "classified_A16.csv",
    "classified_B2.csv", "classified_B4.csv", "classified_B8.csv", "classified_B9.csv",
    "classified_C1.csv", "classified_C2.csv",
    "classified_C3.csv", "classified_C4.csv", "classified_C5.csv"
]

# =========================================================
# Load and Prepare Data
# =========================================================

def load_well_csvs(csv_files):
    dfs = []

    for path in csv_files:
        df = pd.read_csv(path)

        # Infer well name from file
        well_name = os.path.basename(path).replace(".csv", "")
        df["WELL"] = well_name

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def sliding_window_generator(df, feature_cols, label_col, window):
    X, y = [], []

    labels = df[label_col].values

    for i in range(len(df) - window + 1):
        window_df = df.iloc[i:i + window]

        # Majority vote label (noise reduction)
        label_window = labels[i:i + window]
        label = np.bincount(label_window).argmax()



        X.append(window_df[feature_cols].values)
        y.append(label)

    return np.array(X), np.array(y)


def prepare_data(df, window_size):
    feature_cols = [c for c in df.columns if c not in [LABEL_COL, "WELL"]]

    X_all, y_all, wells_all = [], [], []

    for well in df["WELL"].unique():
        df_well = df[df["WELL"] == well].reset_index(drop=True)

        X_w, y_w = sliding_window_generator(
            df_well,
            feature_cols,
            LABEL_COL,
            window_size
        )

        X_all.append(X_w)
        y_all.append(y_w)
        wells_all.extend([well] * len(y_w))

    X = np.vstack(X_all)
    y = np.hstack(y_all)
    wells = np.array(wells_all)

    return X, y, wells, feature_cols


# =========================================================
# Model Components
# =========================================================
def transformer_encoder(inputs, d_model=128, num_heads=4, dff=256, dropout=0.1):
    x = layers.Dense(d_model)(inputs)

    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout
    )(x, x)

    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(dff, activation="relu")(x)
    ffn = layers.Dense(d_model)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    return x


def build_lnuqnet(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    # ---------------- CNN Branch ----------------
    x_cnn = layers.Conv2D(32, (3, 1), padding="same", activation="relu")(inputs)
    x_cnn = layers.Conv2D(64, (3, 1), padding="same", activation="relu")(x_cnn)
    x_cnn = layers.Conv2D(128, (3, 1), padding="same", activation="relu")(x_cnn)
    x_cnn = layers.GlobalAveragePooling2D()(x_cnn)

    # ---------------- Transformer Branch ----------------
    x_tr = layers.Reshape((input_shape[0], input_shape[1]))(inputs)
    x_tr = transformer_encoder(x_tr)
    x_tr = layers.GlobalAveragePooling1D()(x_tr)

    # ---------------- Fusion ----------------
    x = layers.Concatenate()([x_cnn, x_tr])
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model


# =========================================================
# Training Function (Correct)
# =========================================================

def train_lnuqnet(
    model,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    epochs,
    batch_size,
    class_weight
):

    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            "best_lnuqnet_model.keras",
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        shuffle=True,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Loss     : {test_loss:.4f}")
    print(f"Test Accuracy : {test_acc:.4f}")

    return history


# =========================================================
# Main Execution
# =========================================================

if __name__ == "__main__":

    # -------- Load data --------
    df = load_well_csvs(CSV_FILES)

    # -------- Encode labels --------
    le = LabelEncoder()
    df[LABEL_COL] = le.fit_transform(df[LABEL_COL])

    # -------- Prepare sliding windows --------
    X, y, wells, feature_cols = prepare_data(df, WINDOW_SIZE)

    # -------- Normalize features --------
    scaler = StandardScaler()
    X = scaler.fit_transform(
        X.reshape(-1, X.shape[-1])
    ).reshape(X.shape)

    # Add channel dimension for CNN
    X = X[..., np.newaxis]


    TRAIN_WELLS = [
      'classified_C2', 'classified_C4', 'classified_C3',
      'classified_B4', 'classified_A15', 'classified_A16',
      'classified_A10', 'classified_B2', 'classified_B8'
    ]

    TEST_WELLS = [
      'classified_C5', 'classified_B9', 'classified_C1'
    ]

    # -------- Well-based split (EXACT wells) --------
    train_mask = np.isin(wells, TRAIN_WELLS)
    test_mask  = np.isin(wells, TEST_WELLS)

    X_train_full, y_train_full = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    assert not np.any(np.isin(TRAIN_WELLS, TEST_WELLS)), "Overlap between train and test wells!"

    print("Train wells found:", np.unique(wells[train_mask]))
    print("Test wells found:", np.unique(wells[test_mask]))

    assert len(X_train_full) > 0, "Training set is empty!"
    assert len(X_test) > 0, "Test set is empty!"


    # -------- Train / Validation split --------
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        stratify=y_train_full,
        random_state=RANDOM_SEED
    )

    # -------- Class weights --------
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )
    class_weight_dict = dict(enumerate(class_weights))

    # -------- Build model --------
    model = build_lnuqnet(
        input_shape=X_train.shape[1:],
        num_classes=len(le.classes_)
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    model.summary()

        # -------- Train --------
    history = train_lnuqnet(
        model,
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict
    )

### Train and Test Well Distribution

This section prints the list of wells designated for training and testing, ensuring a clear separation and understanding of the data split strategy.

In [ ]:
print("\n=== TRAIN WELLS ===")
for w in sorted(TRAIN_WELLS):
    print(w)

print("\n=== TEST WELLS ===")
for w in sorted(TEST_WELLS):
    print(w)

### Class Distribution Analysis

This utility function and its subsequent calls display the distribution of lithology classes within the training and test datasets.

In [ ]:
import numpy as np
import pandas as pd

def print_class_distribution(y, label_encoder, split_name=""):
    values, counts = np.unique(y, return_counts=True)
    total = len(y)

    print(f"\n=== Class Distribution: {split_name} ===")
    for v, c in zip(values, counts):
        class_name = label_encoder.inverse_transform([v])[0]
        pct = 100 * c / total
        print(f"{str(class_name):20s} | {c:6d} | {pct:6.2f}%")

# Train distribution
print_class_distribution(
    y_train,
    le,
    split_name="TRAIN"
)

# Test distribution
print_class_distribution(
    y_test,
    le,
    split_name="TEST"
)


### Per-Well Training Accuracy

This section evaluates and prints the accuracy of the trained model for each individual well present in the training set. This provides insight into how well the model generalizes across different wells it was trained on.

In [ ]:
from sklearn.metrics import accuracy_score

print("\n=== Per-Well Train Accuracy ===")

for well in np.unique(wells[train_mask]):
    idx = np.where((wells == well) & train_mask)[0]

    X_well = X[idx]
    y_well = y[idx]

    y_pred = np.argmax(model.predict(X_well, verbose=0), axis=1)

    acc = accuracy_score(y_well, y_pred)

    print(f"{well:25s} | Accuracy: {acc:.3f} | Samples: {len(y_well)}")


### Per-Well Test Accuracy

This section evaluates and prints the accuracy of the trained model for each individual well in the held-out test set. This is a crucial metric for assessing the model's generalization capability to unseen wells, which is particularly important in geological applications.

In [ ]:
from sklearn.metrics import accuracy_score

print("\n=== Per-Well Test Accuracy ===")

for well in np.unique(wells[test_mask]):
    idx = np.where((wells == well) & test_mask)[0]

    X_well = X[idx]
    y_well = y[idx]

    y_pred = np.argmax(model.predict(X_well, verbose=0), axis=1)

    acc = accuracy_score(y_well, y_pred)

    print(f"{well:25s} | Accuracy: {acc:.3f} | Samples: {len(y_well)}")


### Feature Shift Visualization

This function and its call generate histograms comparing the distribution of features between the training and test datasets. Visualizing feature shifts helps identify potential data drift or discrepancies that might affect model performance and generalization. It plots the first 5 features by default.

In [ ]:
def plot_feature_shift(X_train, X_test, feature_names, n_features=5):
    import matplotlib.pyplot as plt

    for i in range(min(n_features, len(feature_names))):
        plt.figure()
        plt.hist(
            X_train[:, :, i, 0].ravel(),
            bins=50,
            alpha=0.6,
            density=True,
            label="Train"
        )
        plt.hist(
            X_test[:, :, i, 0].ravel(),
            bins=50,
            alpha=0.6,
            density=True,
            label="Test"
        )
        plt.title(f"Feature Shift: {feature_names[i]}")
        plt.legend()
        plt.show()

plot_feature_shift(
    X_train_full,
    X_test,
    feature_cols
)
